In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from IPython.display import display, Math

keras.utils.set_random_seed(42)
figures = Path("figures")
figures.mkdir(exist_ok=True)
class_names = ["Airplane", "Automobile", "Bird", "Cat", "Deer", "Dog", "Frog", "Horse", "Ship", "Truck"]


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)
indices = [np.where(y_train == c)[0][0] for c in range(10)]
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, idx in zip(axes.ravel(), indices):
    ax.imshow(x_train[idx])
    ax.set_title(class_names[y_train[idx]])
    ax.axis("off")
fig.tight_layout()
fig.savefig(figures / "sample_images.png", dpi=200, bbox_inches="tight")
plt.show()
train_counts = np.bincount(y_train, minlength=10)
test_counts = np.bincount(y_test, minlength=10)
pos = np.arange(10)
width = 0.4
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(pos - width / 2, train_counts, width, label="Training")
ax.bar(pos + width / 2, test_counts, width, label="Testing")
ax.set_xticks(pos, class_names, rotation=35, ha="right")
ax.set_ylabel("Images")
ax.set_title("CIFAR-10 Class Distribution")
ax.legend()
fig.tight_layout()
fig.savefig(figures / "class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
x_train_n = x_train.astype("float32") / 255.0
x_test_n = x_test.astype("float32") / 255.0

display(Math(r"Y(i,j)=\sum_m\sum_n X(i+m,j+n)K(m,n)"))
display(Math(r"N_{\mathrm{out}}=\left\lfloor\frac{N-F+2P}{S}\right\rfloor+1"))
display(Math(r"N_{\mathrm{params}}=(k_h k_w C_{\mathrm{in}}+1)C_{\mathrm{out}}"))

def output_size(n, f, s=1, p=0):
    return (n - f + 2 * p) // s + 1

dimension_table = pd.DataFrame([
    ["3 x 3, Valid, stride 1", output_size(32, 3, 1, 0)],
    ["5 x 5, Valid, stride 1", output_size(32, 5, 1, 0)],
    ["7 x 7, Valid, stride 1", output_size(32, 7, 1, 0)],
    ["3 x 3, Same, stride 1", 32],
    ["3 x 3, Valid, stride 2", output_size(32, 3, 2, 0)],
    ["3 x 3, Same, stride 2", 16]
], columns=["Kernel / Setting", "Output size"])
dimension_table["Output size"] = dimension_table["Output size"].astype(str) + " x " + dimension_table["Output size"].astype(str)
display(dimension_table)


In [ ]:
def build_cnn(pooling="max", first_filters=16):
    pool = layers.MaxPooling2D if pooling == "max" else layers.AveragePooling2D
    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(first_filters, (3, 3), padding="same", activation="relu", name="conv1"),
        pool((2, 2)),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv2"),
        pool((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_cnn()
model.summary()


In [ ]:
start = time.perf_counter()
history = model.fit(x_train_n, y_train, validation_split=0.1, epochs=20, batch_size=32, verbose=1)
training_time = time.perf_counter() - start
probabilities = model.predict(x_test_n, verbose=0)
predictions = np.argmax(probabilities, axis=1)
metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
    "recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
    "f1": f1_score(y_test, predictions, average="weighted", zero_division=0)
}
print(metrics)
print(classification_report(y_test, predictions, target_names=class_names, zero_division=0))


In [ ]:
def plot_curve(key, title, ylabel, filename):
    values = history.history[key]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, len(values) + 1), values, marker="o", markersize=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(figures / filename, dpi=200, bbox_inches="tight")
    plt.show()

plot_curve("accuracy", "Training Accuracy vs Epoch", "Accuracy", "training_accuracy.png")
plot_curve("val_accuracy", "Validation Accuracy vs Epoch", "Accuracy", "validation_accuracy.png")
plot_curve("loss", "Training Loss vs Epoch", "Loss", "training_loss.png")
plot_curve("val_loss", "Validation Loss vs Epoch", "Loss", "validation_loss.png")


In [ ]:
feature_model = keras.Model(model.input, model.get_layer("conv1").output)
feature_maps = feature_model.predict(x_test_n[:1], verbose=0)[0]
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(feature_maps[:, :, i], cmap="gray")
    ax.set_title(f"Feature Map {i + 1}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(figures / "feature_maps.png", dpi=200, bbox_inches="tight")
plt.show()
cm = confusion_matrix(y_test, predictions)
fig, ax = plt.subplots(figsize=(8.5, 7.5))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.xticks(rotation=40, ha="right")
ax.set_title("CNN Confusion Matrix")
fig.tight_layout()
fig.savefig(figures / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
comparison_indices = np.concatenate([np.where(y_train == c)[0][:1000] for c in range(10)])
x_compare = x_train_n[comparison_indices]
y_compare = y_train[comparison_indices]

def train_variant(pooling="max", first_filters=16, epochs=3):
    variant = build_cnn(pooling=pooling, first_filters=first_filters)
    start = time.perf_counter()
    variant.fit(x_compare, y_compare, epochs=epochs, batch_size=64, verbose=0)
    elapsed = time.perf_counter() - start
    pred = np.argmax(variant.predict(x_test_n, verbose=0), axis=1)
    return variant, accuracy_score(y_test, pred), elapsed

max_model, max_accuracy, max_time = train_variant("max", 16)
avg_model, avg_accuracy, avg_time = train_variant("avg", 16)
filter64_model, filter64_accuracy, filter64_time = train_variant("max", 64)
pooling_table = pd.DataFrame([
    ["Max Pooling", "16 x 16 x 16", max_accuracy, max_time],
    ["Average Pooling", "16 x 16 x 16", avg_accuracy, avg_time]
], columns=["Pooling", "First pooled output", "Test accuracy", "Training time (s)"])
filter_table = pd.DataFrame([
    [16, max_model.count_params(), max_accuracy, max_time],
    [64, filter64_model.count_params(), filter64_accuracy, filter64_time]
], columns=["First-layer filters", "Parameters", "Test accuracy", "Training time (s)"])
display(pooling_table)
display(filter_table)


In [ ]:
result_table = pd.DataFrame([
    ["Final training accuracy", history.history["accuracy"][-1]],
    ["Testing accuracy", metrics["accuracy"]],
    ["Weighted precision", metrics["precision"]],
    ["Weighted recall", metrics["recall"]],
    ["Weighted F1-score", metrics["f1"]],
    ["Number of trainable parameters", model.count_params()],
    ["Training time (s)", training_time]
], columns=["Metric", "Value"])
classification_table = pd.DataFrame(classification_report(y_test, predictions, target_names=class_names, output_dict=True, zero_division=0)).T
display(result_table)
display(classification_table)


In [ ]:
display(Math(r"N_{\mathrm{out}}=\left\lfloor\frac{64-5+2(2)}{2}\right\rfloor+1=32"))
display(Math(r"N_{\mathrm{params}}=(3\times3\times3+1)\times64=1792"))
display(Math(r"\mathrm{ReLU}(x)=\max(0,x)"))
display(Math(r"\sigma(x)=\frac{1}{1+e^{-x}}"))

exercise_output = (64 - 5 + 2 * 2) // 2 + 1
exercise_parameters = (3 * 3 * 3 + 1) * 64
activation_table = pd.DataFrame([
    ["ReLU", "max(0, x)", "Fast; sparse activations; avoids saturation for positive inputs"],
    ["Sigmoid", "1 / (1 + exp(-x))", "Bounded output; can saturate and shrink gradients"]
], columns=["Activation", "Expression", "Key effect"])
print("64 x 64, 5 x 5, stride 2, padding 2 ->", exercise_output, "x", exercise_output)
print("64 filters, 3 x 3, RGB input ->", exercise_parameters, "parameters")
display(activation_table)
